In [ ]:
##Instalación de librerías pertinentes

install.packages(c("ISLR2", "glmnet"),repos = "https://cloud.r-project.org")




Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘iterators’, ‘foreach’, ‘shape’, ‘RcppEigen’




In [ ]:
library(ISLR2)
library(glmnet)


Loading required package: Matrix

Loaded glmnet 5.0



In [ ]:
data = data(Credit)
print(data(Credit))
print(head(Credit))
print(dim(Credit))
print(str(Credit))
print(ncol(Credit))
print(nrow(Credit))

[1] "Credit"
   Income Limit Rating Cards Age Education Own Student Married Region Balance
1  14.891  3606    283     2  34        11  No      No     Yes  South     333
2 106.025  6645    483     3  82        15 Yes     Yes     Yes   West     903
3 104.593  7075    514     4  71        11  No      No      No   West     580
4 148.924  9504    681     3  36        11 Yes      No      No   West     964
5  55.882  4897    357     2  68        16  No      No     Yes  South     331
6  80.180  8047    569     4  77        10  No      No      No  South    1151
[1] 400  11
'data.frame':	400 obs. of  11 variables:
 $ Income   : num  14.9 106 104.6 148.9 55.9 ...
 $ Limit    : num  3606 6645 7075 9504 4897 ...
 $ Rating   : num  283 483 514 681 357 569 259 512 266 491 ...
 $ Cards    : num  2 3 4 3 2 4 2 2 5 3 ...
 $ Age      : num  34 82 71 36 68 77 37 87 66 41 ...
 $ Education: num  11 15 11 11 16 10 12 9 13 19 ...
 $ Own      : Factor w/ 2 levels "No","Yes": 1 2 1 2 1 1 2 1 2 2 ...
 $ Student 

**Problema 1: Regresión lineal múltiple.**

In [ ]:
modelo1 <- lm(Balance ~ Income + Limit + Cards + Age, data = Credit)
print(summary(modelo1))


Call:
lm(formula = Balance ~ Income + Limit + Cards + Age, data = Credit)

Residuals:
    Min      1Q  Median      3Q     Max 
-230.51 -115.37  -38.75   37.91  521.33 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -3.996e+02  3.584e+01 -11.150  < 2e-16 ***
Income      -7.493e+00  3.838e-01 -19.522  < 2e-16 ***
Limit        2.628e-01  5.798e-03  45.329  < 2e-16 ***
Cards        2.142e+01  5.953e+00   3.599  0.00036 ***
Age         -8.876e-01  4.811e-01  -1.845  0.06580 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 162.7 on 395 degrees of freedom
Multiple R-squared:  0.876,	Adjusted R-squared:  0.8748 
F-statistic: 697.9 on 4 and 395 DF,  p-value: < 2.2e-16


### Ecuación estimada

Balance_estimado = -399.62 - 7.4931·Income + 0.2628·Limit + 21.4215·Cards - 0.8876·Age

### Interpretación

Income (-7.4931): manteniendo fijos Limit, Cards y Age, un aumento de mil dólares en el ingreso se asocia a una disminución promedio de 7.49 unidades en el saldo. El signo negativo se explica porque el efecto se mide a cupo constante: entre dos personas con el mismo límite de crédito, la de mayor ingreso tiende a mantener un saldo menor.

Limit (0.2628): manteniendo fijos Income, Cards y Age, un dólar adicional de límite de crédito se asocia a un aumento promedio de 0.263 unidades en el saldo.

Para cada coeficiente planteamos H0: βj = 0 (el predictor no tiene asociación con Balance controlando por los demás) versus H1: βj ≠ 0. Con α = 0.05: Income, Limit y Cards tienen p-value < 0.05, por lo que se rechaza H0 y hay evidencia de asociación con Balance. Age tiene p-value = 0.066 > 0.05, por lo que no se rechaza H0 y no hay evidencia suficiente de asociación con Balance.

Call:
lm(formula = Balance \~ Income + Limit + Cards + Age, data = Credit)
Residuals:
Min      1Q  Median      3Q     Max
-230.51 -115.37  -38.75   37.91  521.33
Signif. codes:  0 ‘***’ 0.001 ‘****’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
Residual standard error: 162.7 on 395 degrees of freedom
Multiple R-squared:  0.876,	Adjusted R-squared:  0.8748
F-statistic: 697.9 on 4 and 395 DF,  p-value: < 2.2e-16

El modelo explica el 87.6% de la variabilidad de Balance, con R^2 ajustado de 87.48%. El test F global resultó estadísticamente significativo (F 697.9, con p-value < 2.2 x 10^-16), por lo que existe evidencia de asociación entre el conjunto de predictores y el saldo promedio. Individualmente, Income, Limit y Cards presentaron evidencia de asociación con Balance, mientras que Age no fue significativa al nivel del 5%. El error estándar residual fue de 162.7 unidades de Balance.


### Extra (matemática del código)

In [ ]:
X <- cbind(Intercepto = 1, Income = Credit$Income,
            Limit = Credit$Limit, Cards=Credit$Cards, Age =Credit$Age)
y <- Credit$Balance
beta_gorro <- solve(t(X)%*%X)%*%t(X)%*%y
beta_gorro


Intercepto,-399.6206858
Income,-7.4930547
Limit,0.2628013
Cards,21.4215371
Age,-0.8876401


In [ ]:
y_gorro <- X%*% beta_gorro
residuos <- y -y_gorro
print(head(y_gorro))
print(head(residuos))

          [,1]
[1,]  449.1250
[2,]  543.7209
[3,]  698.6410
[4,] 1014.4566
[5,]  451.0739
[6,] 1131.6860
           [,1]
[1,] -116.12499
[2,]  359.27914
[3,] -118.64105
[4,]  -50.45663
[5,] -120.07388
[6,]   19.31400


In [ ]:
SSE <- sum(residuos^2)
SST <- sum((y-mean(y))^2)
R2 <- 1 - SSE/SST
n <- nrow(X)
p <- ncol(X) - 1
R2_ajustado <- 1-(1-R2) * (n-1) / (n-p-1)
R2
R2_ajustado

[1] 0.8760446

[1] 0.8747894

In [ ]:
sigma2_gorro <- SSE / (n - p - 1)

Var_beta_gorro <- sigma2_gorro * solve(t(X) %*% X)

error_estandar <- sqrt(diag(Var_beta_gorro))

t_value <- as.vector(beta_gorro) / error_estandar

p_value <- 2 * pt(
  -abs(t_value),
  df = n - p - 1
)

tabla_manual <- data.frame(
  Estimate = as.vector(beta_gorro),
  Std_Error = error_estandar,
  t_value = t_value,
  p_value = p_value
)

print(tabla_manual)

               Estimate    Std_Error    t_value       p_value
Intercepto -399.6206858 35.839268713 -11.150358  2.742602e-25
Income       -7.4930547  0.383831055 -19.521752  6.698783e-60
Limit         0.2628013  0.005797683  45.328673 1.317855e-158
Cards        21.4215371  5.952548362   3.598717  3.604316e-04
Age          -0.8876401  0.481132869  -1.844896  6.580110e-02


In [ ]:
SSR <- SST - SSE
F_global <- (SSR/p) / (SSE/(n-p-1))

p_F <-pf(F_global,
         df1=p,
         df2 =n-p-1,
         lower.tail = FALSE
         )
print(F_global)
print(p_F)

[1] 697.9076
[1] 1.447132e-177


**Rechazamos H0: Existe evidencia de que el conjunto de variables Income, Limit, Cards y Age tiene una asociación estadísticamente significativa con Balance.**

**Problema 2: Selección de modelos.**

In [ ]:
modelo_completo <- lm(Balance ~., data = Credit)

### 2.1 Comparación con el modelo del Problema 1

In [ ]:
modelo1 <- lm(Balance ~ Income + Limit + Cards + Age, data = Credit)

comparacion <- data.frame(
  Modelo = c("Problema 1 (4 var)", "Completo (todas)"),
  R2_aj  = c(summary(modelo1)$adj.r.squared,
             summary(modelo_completo)$adj.r.squared),
  AIC    = c(AIC(modelo1), AIC(modelo_completo))
)
comparacion

Modelo,R2_aj,AIC
<chr>,<dbl>,<dbl>
Problema 1 (4 var),0.8747894,5215.578
Completo (todas),0.9538287,4823.370


El modelo completo presenta un R^2 adj. de 0.9538 frente a 0.8748 del modelo del problema 1, y un AIC de 4823.4 frente a 5215.6. Como el R^2 adj es mayor y el AIC menor, ambos criterios coinciden en preferir el modelo completo. La mejora es sustancial y no se debe únicamente al mayor número de predictores: el r^2 adj penaliza la inclusión de variables irrelevantes, por lo que su aumento indica una ganancia real del ajuste. Preferimos entonces el modelo con todas las variables.

### Extra (matemática del código)



In [ ]:
X_all <- model.matrix(Balance ~ ., data = Credit)
y <- Credit$Balance
n_all <- nrow(X_all)
p_all <- ncol(X_all) - 1

beta_all <- solve(t(X_all) %*% X_all) %*% (t(X_all) %*% y)
y_hat_all <- X_all %*% beta_all
e_all <- y - y_hat_all

SSE_all <- sum(e_all^2)
SST <- sum((y - mean(y))^2)
R2_all <- 1 - SSE_all/SST
R2_adj_all <- 1 - (1 - R2_all) * (n_all - 1) / (n_all - p_all - 1)

AIC_manual <- n_all * log(SSE_all/n_all) + 2*(p_all + 2) + n_all * (1 + log(2*pi))

c(R2_adj_all = R2_adj_all, AIC_manual = AIC_manual)

R2_adj_all   AIC_manual 
   0.9538287 4823.3703910

### 2.2 Backward por AIC

In [ ]:
modelo_backward <- step(modelo_completo, direction = "backward", trace = 1)
summary(modelo_backward)

Start:  AIC=3686.22
Balance ~ Income + Limit + Rating + Cards + Age + Education + 
    Own + Student + Married + Region

            Df Sum of Sq      RSS    AIC
- Region     2     14084  3800814 3683.7
- Education  1      4615  3791345 3684.7
- Married    1      6619  3793349 3684.9
- Own        1     11269  3798000 3685.4
<none>                    3786730 3686.2
- Age        1     42558  3829288 3688.7
- Rating     1     52314  3839044 3689.7
- Cards      1    162702  3949432 3701.0
- Limit      1    331050  4117780 3717.7
- Student    1   6326012 10112742 4077.1
- Income     1  10831162 14617892 4224.5

Step:  AIC=3683.7
Balance ~ Income + Limit + Rating + Cards + Age + Education + 
    Own + Student + Married

            Df Sum of Sq      RSS    AIC
- Married    1      4545  3805359 3682.2
- Education  1      4757  3805572 3682.2
- Own        1     10760  3811574 3682.8
<none>                    3800814 3683.7
- Age        1     45650  3846464 3686.5
- Rating     1     49473  3850


Call:
lm(formula = Balance ~ Income + Limit + Rating + Cards + Age + 
    Student, data = Credit)

Residuals:
    Min      1Q  Median      3Q     Max 
-170.00  -77.85  -11.84   56.87  313.52 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -493.73419   24.82476 -19.889  < 2e-16 ***
Income        -7.79508    0.23342 -33.395  < 2e-16 ***
Limit          0.19369    0.03238   5.981 4.98e-09 ***
Rating         1.09119    0.48480   2.251   0.0250 *  
Cards         18.21190    4.31865   4.217 3.08e-05 ***
Age           -0.62406    0.29182  -2.139   0.0331 *  
StudentYes   425.60994   16.50956  25.780  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 98.61 on 393 degrees of freedom
Multiple R-squared:  0.9547,	Adjusted R-squared:  0.954 
F-statistic:  1380 on 6 and 393 DF,  p-value: < 2.2e-16


In [ ]:
data.frame(
  Modelo = c("Problema 1", "Completo", "Backward AIC"),
  R2_aj  = c(summary(modelo1)$adj.r.squared,
             summary(modelo_completo)$adj.r.squared,
             summary(modelo_backward)$adj.r.squared),
  AIC    = c(AIC(modelo1), AIC(modelo_completo), AIC(modelo_backward))
)

Modelo,R2_aj,AIC
<chr>,<dbl>,<dbl>
Problema 1,0.8747894,5215.578
Completo,0.9538287,4823.370
Backward AIC,0.9539961,4817.039


El procedimiento backward seleccionó el modelo Balance ~ Income + Limit + Rating + Cards + Age + Student, eliminando Region, Married, Education y Own. Respecto del modelo completo, el R²aj se mantiene prácticamente igual (0.9540 frente a 0.9538) mientras el AIC disminuye de 4823.4 a 4817.0, es decir, se obtiene el mismo poder explicativo con cuatro predictores menos.

Las variables descartadas son las demográficas y geográficas, que no aportan información adicional una vez controladas las variables financieras.

Cabe notar que el criterio AIC mantiene simultáneamente a Limit y Rating pese a su altísima correlación.


**Problema 3: Multicolinealidad.**

### 3.1 Matriz de correlaciones

In [ ]:
num <- Credit[, c("Income","Limit","Rating","Cards","Age","Education","Balance")]
round(cor(num), 3)

,Income,Limit,Rating,Cards,Age,Education,Balance
Income,1.000,0.792,0.791,-0.018,0.175,-0.028,0.464
Limit,0.792,1.000,0.997,0.010,0.101,-0.024,0.862
Rating,0.791,0.997,1.000,0.053,0.103,-0.030,0.864
Cards,-0.018,0.010,0.053,1.000,0.043,-0.051,0.086
Age,0.175,0.101,0.103,0.043,1.000,0.004,0.002
Education,-0.028,-0.024,-0.030,-0.051,0.004,1.000,-0.008
Balance,0.464,0.862,0.864,0.086,0.002,-0.008,1.000


¿Qué pares de predictores
presentan las correlaciones más altas?

Las correlaciones más altas se dan entre Limit y Rating ( r = 0.997), un valor casi perfecto que indica que ambas variables entregan prácticamente la misma información sobre el cliente. Le siguen income-Limit (r = 0.792) e income-rating (r=0.791). El resto de los predictores (Cards, Age, Education) presenta correlaciones cercanas a cero entre sí y con las demás variables.

### 3.2 VIF

In [ ]:
predictores <- c("Income","Limit","Rating","Cards","Age","Education")
vif_manual <- sapply(predictores, function(v) {
  f <- as.formula(paste(v, "~", paste(setdiff(predictores, v), collapse = " + ")))
  1 / (1 - summary(lm(f, data = Credit))$r.squared)
})
round(vif_manual, 3)

Income     Limit    Rating     Cards       Age Education 
    2.773   228.848   230.613     1.434     1.039     1.008

¿Qué variables presentan los valores más altos de VIF? ¿Existe evidencia de problemas de multicolinealidad?

Los VIF obtenidos son: Income 2.77, Limit 228.85, Rating 230.61, Cards 1.43, Age 1.04 y Education 1.01.

Rating y limit presentan valores extremadamente altos, muy por encima del umbral habitual de 10, por lo que existe evidencia clara de multicolinealidad severa. Este problema se origina casi por completo en la altísima correlación entre limit y rating. Income muestra un VIF moderado (2.77), aceptable, mientras que Cards, Age y education no presentan indicios de multicolinealidad.

### 3.3
Explique brevemente por qué la presencia de predictores altamente correlacionados puede dificultar la
interpretación de los coeficientes de un modelo de regresión.

si dos predictores se mueven casi juntos, los datos no permiten separar su efecto individual; la varianza de los Beta se infla, los errores estándar crecen, los t caen y los coeficientes se vuelven inestables (cambian mucho al agregar o quitar una variable), incluso con R² alto.

**Problema 4: Predicción y regularización.**

### 4.1 Investigue brevemente en qué consiste la validación cruzada K-fold y explique con sus propias palabras cómo permite estimar el error predictivo.

Los datos se dividen al azar en K subconjuntos (*folds*) de tamaño similar. El
procedimiento se repite K veces: en la iteración k se ajusta el modelo usando los
K - 1 folds restantes (**entrenamiento**) y se predicen las observaciones del fold $k$
(**validación**), que no participaron del ajuste. Se calcula el error en ese fold y al
final se promedian los K errores.

La clave es que **cada observación se predice exactamente una vez con un modelo que no la
vio**. Eso evita el optimismo del error de entrenamiento y entrega una estimación más real del error que cometeríamos con un individuo
nuevo. Comparado con partir en una sola muestra de entrenamiento y prueba, K-fold usa todos
los datos para entrenar y para validar, lo que reduce la varianza de la estimación. Además,
todo el proceso de modelamiento (incluida la selección de variables) debe ejecutarse
**dentro** de cada fold; si se selecciona con todos los datos y luego se valida, el error
resulta artificialmente bajo.

### 4.2 Validación cruzada: modelo lineal completo

In [ ]:
n <- nrow(Credit)
K <- 5

set.seed(123)
folds <- sample(rep(1:K, length.out = n))

rmse <- function(actual, pred) sqrt(mean((actual - pred)^2))

rmse_lineal <- numeric(K)
for (k in 1:K) {
  train <- Credit[folds != k, ]
  test  <- Credit[folds == k, ]
  modelo_k <- lm(Balance ~ ., data = train)
  pred <- predict(modelo_k, newdata = test)
  rmse_lineal[k] <- rmse(test$Balance, pred)
}

round(rmse_lineal, 2)
round(mean(rmse_lineal), 2)

[1] 101.25  89.70 101.34  96.38 113.52

[1] 100.44

El RMSE promedio del modelo lineal completo en los 5 folds fue de 100.44 (valores por fold entre 89.70 y 113.52). Este número estima el error típico al predecir el Balance de un cliente que no fue usado para ajustar el modelo.

### 4.3 Selección mediante AIC dentro de cada fold

In [ ]:
rmse_aic <- numeric(K)
n_predictores_aic <- numeric(K)
for (k in 1:K) {
  train <- Credit[folds != k, ]
  test  <- Credit[folds == k, ]
  modelo_completo_k <- lm(Balance ~ ., data = train)
  modelo_aic_k <- step(modelo_completo_k, direction = "backward", trace = 0)
  pred <- predict(modelo_aic_k, newdata = test)
  rmse_aic[k] <- rmse(test$Balance, pred)
  n_predictores_aic[k] <- length(coef(modelo_aic_k)) - 1
}

round(rmse_aic, 2)
round(mean(rmse_aic), 2)
n_predictores_aic

[1] 100.48  89.11 102.84  95.99 111.60

[1] 100

[1] 6 6 5 7 6

El RMSE promedio con selección AIC fue de 100.00, prácticamente igual al del modelo completo. La selección backward mantuvo casi siempre las mismas variables (Income, Limit, Rating, Cards y Student en los 5 folds; Age en 4 de los 5), usando entre 5 y 7 predictores según el fold. Es importante repetir la selección dentro de cada fold usando solo los datos de entrenamiento, para que el RMSE no resulte artificialmente bajo.

### 4.4 Ridge

Reutilizamos la misma partición (`folds`) y la función `rmse()` definidas en 4.2, evaluando ahora regresión Ridge (α = 0) con λ elegido por validación cruzada interna mediante `cv.glmnet()`.

In [ ]:
X <- model.matrix(Balance ~ ., data = Credit)[, -1]   # dummies, sin intercepto
Y <- Credit$Balance

rmse_ridge <- numeric(K)
lambdas    <- numeric(K)

set.seed(123)
for (k in 1:K) {
  Xe <- X[folds != k, ];  ye <- Y[folds != k]   # entrenamiento
  Xv <- X[folds == k, ];  yv <- Y[folds == k]   # validación

  cv <- cv.glmnet(Xe, ye, alpha = 0)            # alpha = 0  ->  ridge
  lambdas[k] <- cv$lambda.min

  pred <- predict(cv, newx = Xv, s = "lambda.min")
  rmse_ridge[k] <- rmse(yv, pred)
}

round(rmse_ridge, 2)
mean(rmse_ridge)
round(lambdas, 2)

[1] 118.09 136.30 119.70 110.71 109.71

[1] 118.8996

[1] 40.42 39.36 40.13 39.79 38.57

El RMSE promedio de Ridge fue de 118.90, peor que el modelo lineal (100.44) y que la selección AIC (100.00). λmín es el valor de penalización que, según la validación cruzada interna de `cv.glmnet()`, minimiza el error de predicción: un λ más grande encoge más los coeficientes hacia cero (baja la varianza pero sube el sesgo), mientras que uno más chico se acerca a mínimos cuadrados. En este caso λmín estuvo entre 38.57 y 40.42 según el fold.

### 4.5 Comparación de los tres procedimientos

In [ ]:
tabla_rmse <- data.frame(
  Procedimiento = c("Modelo lineal completo", "Selección mediante AIC", "Regresión Ridge"),
  RMSE_promedio  = round(c(mean(rmse_lineal), mean(rmse_aic), mean(rmse_ridge)), 2)
)
tabla_rmse

Procedimiento,RMSE_promedio
<chr>,<dbl>
Modelo lineal completo,100.44
Selección mediante AIC,100.00
Regresión Ridge,118.90


Para predecir el Balance de un nuevo cliente usaríamos el modelo con selección por AIC: tiene el RMSE promedio más bajo (100.00) y usa menos predictores que el modelo completo, cuyo RMSE (100.44) es prácticamente igual. La diferencia entre ambos es pequeña frente a la variación del RMSE entre folds (entre 89.70 y 113.52), por lo que en la práctica son equivalentes en poder predictivo. Ridge quedó claramente peor (118.90): aquí n es mucho mayor que p y la relación entre los predictores y Balance es fuerte (R² ≈ 0.95), de modo que encoger los coeficientes hacia cero solo agrega sesgo sin una ganancia real en varianza.

ridge

In [ ]:
Z  <- scale(X)              # predictores estandarizados
yc <- Y - mean(Y)
I  <- diag(ncol(Z))

ridge <- function(lambda) solve(t(Z) %*% Z + lambda * I) %*% (t(Z) %*% yc)

trayectoria <- sapply(c(0, 1, 10, 100, 1000), ridge)
dimnames(trayectoria) <- list(colnames(Z), paste0("lambda=", c(0, 1, 10, 100, 1000)))
round(trayectoria, 1)

,lambda=0,lambda=1,lambda=10,lambda=100,lambda=1000
Income,-275.0,-271.6,-242.4,-94.5,23.0
Limit,440.7,367.5,302.3,211.5,84.1
Rating,175.8,245.6,281.4,209.7,84.2
Cards,24.3,21.4,20.2,22.4,10.3
Age,-10.6,-10.9,-13.0,-19.1,-5.7
Education,-3.4,-3.0,-2.3,-0.4,-0.1
OwnYes,-5.3,-5.2,-4.3,0.2,2.0
StudentYes,127.9,127.0,123.1,97.7,34.0
MarriedYes,-4.2,-4.7,-5.4,-6.0,-2.0
RegionSouth,5.1,5.1,5.0,3.5,0.3


**Conclusión**

Las variables más relevantes para explicar Balance fueron Income, Limit, Rating, Cards y Student, estables en la selección por AIC en los cinco folds. Existe evidencia clara de multicolinealidad entre Limit y Rating (VIF sobre 228), lo que dificulta interpretar sus coeficientes por separado, aunque no afecta la capacidad predictiva del modelo. En la validación cruzada, el modelo lineal completo y la selección por AIC tuvieron un desempeño prácticamente igual (RMSE promedio 100.44 y 100.00), mientras que Ridge resultó peor (118.90): con n mucho mayor que p y señal fuerte, penalizar los coeficientes solo agregó sesgo sin reducir la varianza.